# 152 Machine Translation (EN -> FR)

Neural Machine Translation with the **encoder-decoder Transformer** from
`151_transformer_code.py`.

**Pipeline**
1. Build source / target vocabs (+ `PAD/BOS/EOS/UNK`)
2. Encode a tiny parallel corpus -- **track tensor shapes**
3. Teacher-forced training (predict next FR token)
4. Greedy decode one demo pair
5. Plot **decoder cross-attention** (FR queries x EN keys)

Companion: `150_transformer.py` (intuition), `151_transformer_code.py` (classes).


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from importlib import import_module

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F_torch

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

t151 = import_module("151_transformer_code")
Seq2SeqTransformer = t151.Seq2SeqTransformer
make_pad_mask = t151.make_pad_mask
make_causal_mask = t151.make_causal_mask
show = t151.show
PAD_ID = t151.PAD_ID  # 0

FS_FIGURE_TITLE, FS_AX_TITLE, FS_AXIS, FS_ANNOT = 28, 22, 18, 12
FIGURE_DPI, SCREEN_FRACTION = 150, 0.95
plt.rcParams.update({
    "figure.titlesize": FS_FIGURE_TITLE,
    "axes.titlesize": FS_AX_TITLE,
    "axes.labelsize": FS_AXIS,
    "xtick.labelsize": FS_AXIS,
    "ytick.labelsize": FS_AXIS,
    "figure.dpi": FIGURE_DPI,
})

def screen_figsize(fraction=SCREEN_FRACTION, dpi=FIGURE_DPI):
    try:
        import tkinter as tk
        root = tk.Tk(); root.withdraw()
        w, h = root.winfo_screenwidth(), root.winfo_screenheight()
        root.destroy()
        return (w / dpi * fraction, h / dpi * fraction)
    except Exception:
        return (16 * fraction / 0.95, 9 * fraction / 0.95)

def show_maximized(fig):
    fig.set_dpi(FIGURE_DPI)
    fig.set_size_inches(screen_figsize(), forward=True)
    fig.canvas.draw_idle()
    manager = getattr(fig.canvas, "manager", None)
    if manager is not None:
        try:
            manager.window.state("zoomed")
        except Exception:
            try:
                manager.window.showMaximized()
            except Exception:
                pass
    plt.show()

print("Imported 151 classes OK. PAD_ID =", PAD_ID)


## Parallel corpus + special tokens

Source (EN) goes to the **encoder**.
Target (FR) is wrapped with `<bos> ... <eos>` for the **decoder**.

Teacher forcing:
- decoder input = `tgt[:, :-1]`  (`<bos>` + tokens) → shape `(B, T-1)`
- labels        = `tgt[:, 1:]`   (tokens + `<eos>`) → shape `(B, T-1)`


In [ ]:
PAIRS = [
    ("the cat sat on the mat", "le chat s assit sur le tapis"),
    ("the dog sat on the mat", "le chien s assit sur le tapis"),
    ("the cat eats fish", "le chat mange du poisson"),
    ("the dog eats meat", "le chien mange de la viande"),
    ("I love cats", "j aime les chats"),
    ("I love dogs", "j aime les chiens"),
    ("she reads a book", "elle lit un livre"),
    ("he reads a book", "il lit un livre"),
    ("we drink water", "nous buvons de l eau"),
    ("they drink water", "ils boivent de l eau"),
]

DEMO_EN = "the cat eats fish"
DEMO_FR = "le chat mange du poisson"

BOS_ID, EOS_ID, UNK_ID = 1, 2, 3
SPECIAL = {0: "<pad>", 1: "<bos>", 2: "<eos>", 3: "<unk>"}


class Vocab:
    def __init__(self, sentences):
        words = sorted({w for s in sentences for w in s.split()})
        self.itos = dict(SPECIAL)
        for i, w in enumerate(words, start=4):
            self.itos[i] = w
        self.stoi = {w: i for i, w in self.itos.items()}

    def __len__(self):
        return len(self.itos)

    def encode(self, sentence, add_bos, add_eos):
        ids = [self.stoi.get(w, UNK_ID) for w in sentence.split()]
        if add_bos:
            ids = [BOS_ID] + ids
        if add_eos:
            ids = ids + [EOS_ID]
        return ids

    def decode(self, ids, skip_special=True):
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        toks = []
        for i in ids:
            if skip_special and i in (PAD_ID, BOS_ID, EOS_ID):
                continue
            toks.append(self.itos.get(int(i), "<unk>"))
        return " ".join(toks)


def pad_batch(seqs, pad_id=PAD_ID):
    # list[list[int]] -> (B, L_max)
    B, L = len(seqs), max(len(s) for s in seqs)
    out = torch.full((B, L), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, : len(s)] = torch.tensor(s, dtype=torch.long)
    return out


src_vocab = Vocab([en for en, _ in PAIRS])
tgt_vocab = Vocab([fr for _, fr in PAIRS])
src_seqs = [src_vocab.encode(en, add_bos=False, add_eos=True) for en, _ in PAIRS]
tgt_seqs = [tgt_vocab.encode(fr, add_bos=True, add_eos=True) for _, fr in PAIRS]
src_batch = pad_batch(src_seqs)  # (N, S)
tgt_batch = pad_batch(tgt_seqs)  # (N, T)

print(f"V_src={len(src_vocab)}  V_tgt={len(tgt_vocab)}")
show("src_batch", src_batch)
show("tgt_batch", tgt_batch)
print("src[0]:", src_vocab.decode(src_batch[0]))
print("tgt[0]:", tgt_vocab.decode(tgt_batch[0]))


## Model + teacher-forced loss

Uses `Seq2SeqTransformer` from 151 (configurable vocab / width / depth).

Cross-entropy ignores `PAD`. After a forward pass, each
`MultiHeadAttention.last_attn_weights` holds `(B, H, Lq, Lk)`.


In [ ]:
def teacher_forced_loss(model, src, tgt, track=False):
    # src: (B, S)   tgt: (B, T) with BOS...EOS
    if track:
        print("\n=== TEACHER FORCING SHAPES ===")
        show("src (encoder input)", src)
        show("tgt (full with BOS/EOS)", tgt)
    tgt_in = tgt[:, :-1]   # (B, T-1)
    labels = tgt[:, 1:]    # (B, T-1)
    if track:
        show("tgt_in  (decoder in)", tgt_in)
        show("labels (next tokens)", labels)
    logits = model(src, tgt_in, track=track)  # (B, T-1, V)
    if track:
        show("logits", logits)
    V = logits.size(-1)
    return F_torch.cross_entropy(
        logits.reshape(-1, V), labels.reshape(-1), ignore_index=PAD_ID
    )


@torch.no_grad()
def greedy_decode(model, src, bos_id=BOS_ID, eos_id=EOS_ID, max_len=20, track=False):
    # src: (1, S) -> ys: (1, T_out)
    model.eval()
    if track:
        print("\n=== GREEDY DECODE ===")
        show("src", src)
    src_mask = make_pad_mask(src)                       # (1, 1, 1, S)
    memory = model.encoder(src, src_mask, track=track)  # (1, S, D)
    if track:
        show("memory", memory)
    ys = torch.tensor([[bos_id]], dtype=torch.long, device=src.device)  # (1, 1)
    for step in range(max_len):
        tgt_mask = make_pad_mask(ys) & make_causal_mask(ys.size(1), ys.device)
        logits = model.decoder(ys, memory, tgt_mask, src_mask, track=False)
        next_id = logits[:, -1, :].argmax(-1, keepdim=True)  # (1, 1)
        ys = torch.cat([ys, next_id], dim=1)                 # (1, t+1)
        if track:
            print(f"  step {step}: id={int(next_id)} -> ys {tuple(ys.shape)}")
        if int(next_id) == eos_id:
            break
    if track:
        show("decoded ids", ys)
    return ys


torch.manual_seed(0)
d_model, n_heads, d_ff, n_layers = 64, 4, 128, 2
model = Seq2SeqTransformer(
    src_vocab=len(src_vocab),
    tgt_vocab=len(tgt_vocab),
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    n_enc=n_layers,
    n_dec=n_layers,
)
print("params:", sum(p.numel() for p in model.parameters()))


## Train on the toy corpus (overfit is OK for this lecture demo)


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
model.train()
epochs = 400
for ep in range(1, epochs + 1):
    opt.zero_grad()
    loss = teacher_forced_loss(model, src_batch, tgt_batch, track=False)
    loss.backward()
    opt.step()
    if ep == 1 or ep % 50 == 0 or ep == epochs:
        print(f"epoch {ep:4d}  loss={loss.item():.4f}")


## Demo pair: track shapes + read cross-attention

After the forward, `model.cross_attn_weights(layer=-1)` returns
`(B, H, T_in, S)` from the last decoder layer's **cross-attention**.


In [ ]:
demo_src = pad_batch([src_vocab.encode(DEMO_EN, add_bos=False, add_eos=True)])
demo_tgt = pad_batch([tgt_vocab.encode(DEMO_FR, add_bos=True, add_eos=True)])
model.eval()
with torch.no_grad():
    _ = teacher_forced_loss(model, demo_src, demo_tgt, track=True)

cross = model.cross_attn_weights(layer=-1)   # (1, H, T-1, S)
show("cross-attn last layer", cross)
cross_mean = cross[0].mean(0)                # (T-1, S)
show("cross-attn mean heads", cross_mean)

tgt_in_ids = demo_tgt[0, :-1].tolist()
src_ids = demo_src[0].tolist()
s_len = int((demo_src[0] != PAD_ID).sum())
t_len = int((demo_tgt[0, :-1] != PAD_ID).sum())
src_toks = [src_vocab.itos[i] for i in src_ids[:s_len]]
tgt_toks = [tgt_vocab.itos[i] for i in tgt_in_ids[:t_len]]
attn_mean = cross_mean[:t_len, :s_len].cpu()
attn_heads = cross[0, :, :t_len, :s_len].cpu()

print("EN:", DEMO_EN)
print("FR:", DEMO_FR)
print("src tokens:", src_toks)
print("tgt query tokens:", tgt_toks)

with torch.no_grad():
    hyp = greedy_decode(model, demo_src, track=True)
print("greedy:", tgt_vocab.decode(hyp[0]))
print("ref:   ", DEMO_FR)


## Attention matrices for the translation pair

Rows = French **query** tokens (decoder).
Columns = English **key** tokens (encoder memory).


In [ ]:
def plot_cross_attention(weights, src_tokens, tgt_tokens, title):
    if weights.ndim == 2:
        fig, ax = plt.subplots(figsize=screen_figsize(), dpi=FIGURE_DPI)
        fig.suptitle(title, fontsize=FS_FIGURE_TITLE, fontweight="bold")
        im = ax.imshow(weights.numpy(), cmap="viridis", aspect="auto", vmin=0, vmax=1)
        ax.set_xticks(range(len(src_tokens)))
        ax.set_yticks(range(len(tgt_tokens)))
        ax.set_xticklabels(src_tokens, rotation=45, ha="right")
        ax.set_yticklabels(tgt_tokens)
        ax.set_xlabel("source (EN) keys", fontsize=FS_AXIS)
        ax.set_ylabel("target (FR) queries", fontsize=FS_AXIS)
        for i in range(weights.size(0)):
            for j in range(weights.size(1)):
                ax.text(
                    j, i, f"{weights[i, j]:.2f}", ha="center", va="center",
                    color="white" if weights[i, j] > 0.4 else "black", fontsize=9,
                )
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.text(
            0.5, 0.02,
            "Bright cell => that EN token strongly informs this FR token.",
            ha="center", fontsize=FS_ANNOT,
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.95),
        )
        fig.tight_layout(rect=[0, 0.08, 1, 1])
        show_maximized(fig)
        return

    H = weights.size(0)
    ncols = min(4, H)
    nrows = int(np.ceil(H / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=screen_figsize(), dpi=FIGURE_DPI)
    fig.suptitle(title, fontsize=FS_FIGURE_TITLE - 2, fontweight="bold")
    axes = np.array(axes).reshape(-1)
    for h in range(H):
        ax = axes[h]
        im = ax.imshow(weights[h].numpy(), cmap="viridis", aspect="auto", vmin=0, vmax=1)
        ax.set_xticks(range(len(src_tokens)))
        ax.set_yticks(range(len(tgt_tokens)))
        ax.set_xticklabels(src_tokens, rotation=45, ha="right", fontsize=FS_ANNOT)
        ax.set_yticklabels(tgt_tokens, fontsize=FS_ANNOT)
        ax.set_title(f"head {h}", fontsize=FS_AX_TITLE - 4)
    for h in range(H, len(axes)):
        axes[h].axis("off")
    fig.colorbar(im, ax=axes[:H].tolist(), fraction=0.02, pad=0.02)
    fig.subplots_adjust(top=0.88, bottom=0.12, wspace=0.35, hspace=0.45)
    show_maximized(fig)


plot_cross_attention(
    attn_mean, src_toks, tgt_toks,
    title=f"Cross-attention (mean heads): '{DEMO_EN}' -> '{DEMO_FR}'",
)
plot_cross_attention(
    attn_heads, src_toks, tgt_toks,
    title=f"Cross-attention per head: '{DEMO_EN}' -> '{DEMO_FR}'",
)


## Takeaway

- Encoder writes English **memory** `(B, S, D)`.
- Decoder generates French left-to-right; **cross-attention** reads that memory.
- The heatmap is exactly the softmax attention weights `(T, S)` (mean over heads, or per head).
